# 1. MHA、MQA、GQA

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

d:\pycode\leetcode_exp\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
class MHA(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
    
    def forward(self, q, k, v, mask=None):

        batch_size = q.size(0)

        q = self.q_proj(q).view(batch_size, -1 ,self.num_heads, self.head_dim).transpose(1,2)
        k = self.k_proj(k).view(batch_size, -1 ,self.num_heads, self.head_dim).transpose(1,2)
        v = self.v_proj(v).view(batch_size, -1 ,self.num_heads, self.head_dim).transpose(1,2)

        scores = torch.matmul(q, k.transpose(-2,-1)/math.sqrt(self.head_dim))

        if mask is not None:
            scores = scores.masked_fill(mask, -float('inf'))
        attn_weights = F.softmax(scores, dim = -1)
        attn_weights = self.dropout(attn_weights)

        out = torch.matmul(attn_weights, v)

        out = out.transpose(1,2).contiguous().view(batch_size, -1, self.d_model)

        out = self.out_proj(out)

        return out


In [ ]:
batch_size = 2
num_heads = 8
seq_len = 16
d_model = 512
x = torch.randn(batch_size, seq_len, d_model)
mha = MHA(d_model, num_heads)

mha_out = mha(x, x, x)
print(mha_out)
print(mha_out.shape)


tensor([[[-0.1392,  0.1014,  0.2099,  ..., -0.1294, -0.0160,  0.2766],
         [-0.1187,  0.1298,  0.1177,  ..., -0.1087,  0.0171,  0.2122],
         [-0.0956,  0.1481,  0.1536,  ..., -0.0867,  0.0124,  0.2000],
         ...,
         [-0.0756,  0.1208,  0.1185,  ..., -0.0905,  0.0492,  0.1895],
         [-0.0955,  0.0049,  0.1599,  ..., -0.1010,  0.0352,  0.0972],
         [-0.1323,  0.1356,  0.1450,  ..., -0.0534, -0.0113,  0.1812]],

        [[ 0.0440,  0.0784, -0.0327,  ..., -0.0567,  0.0675, -0.0444],
         [ 0.0672,  0.1120, -0.1094,  ..., -0.0694, -0.0011, -0.0620],
         [ 0.0670,  0.1108, -0.0201,  ..., -0.1549, -0.0430, -0.0243],
         ...,
         [ 0.0996,  0.0994, -0.0628,  ..., -0.0438, -0.1135, -0.0237],
         [ 0.0563,  0.0082, -0.0468,  ..., -0.0292, -0.0322,  0.0180],
         [ 0.0750,  0.0597, -0.1006,  ..., -0.1485, -0.0614,  0.0121]]],
       grad_fn=<ViewBackward0>)
torch.Size([2, 16, 512])


# 2. LayerNorm/RMSNorm

## 2.1 LayerNorm

In [1]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, features, eps = 1e-6):
        super(LayerNorm, self).__init__()

        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps
    
    def forward(self, x):
        mean = x.mean(-1, keepdim = True)
        std = x.std(-1, keepdim = True, unbiased = False)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

features = 512
x = torch.randn(2, 10 ,features)

layernorm = LayerNorm(features)

layernorm(x)

/Users/dullyang/Documents/pycode/leetcode_exp/.venv/lib/python3.9/site-packages/torch/_subclasses/functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


tensor([[[-0.2348,  0.8162,  0.2438,  ...,  0.2811, -0.2698, -1.0907],
         [-0.3335, -0.0265,  0.8451,  ...,  0.7611,  0.2625, -1.8678],
         [ 0.2699, -1.2843,  0.1513,  ...,  1.3001,  0.6041, -0.3376],
         ...,
         [-0.1310, -0.4900, -1.9740,  ..., -1.5368,  0.1293, -1.2749],
         [-1.1287, -0.6404, -1.2999,  ...,  0.7292,  0.9573, -1.3617],
         [-0.7729, -3.1812,  1.2069,  ..., -1.7597,  0.5046, -0.5979]],

        [[ 0.4768, -0.4366,  0.9638,  ..., -0.0034,  1.1211, -0.3811],
         [-0.7350, -0.7611, -2.1791,  ..., -0.3949, -0.1366, -1.1457],
         [-1.6265, -0.3403, -0.4720,  ...,  0.5718, -1.0878, -0.4516],
         ...,
         [ 0.1124,  0.4878, -0.7060,  ...,  1.9821, -0.8511,  0.1694],
         [ 1.7885, -1.2641, -0.2146,  ...,  0.9574,  0.7372, -1.7198],
         [-0.9803,  0.6547,  0.4883,  ..., -1.4095,  1.0065,  0.2186]]],
       grad_fn=<AddBackward0>)

## 2.2 RMSNorm

In [8]:
import torch
import torch.nn as nn
import math

class RMSNorm(nn.Module):
    def __init__(self, features, eps = 1e-6):
        super(RMSNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.eps = eps
    
    def forward(self, x):
        return self.gamma * x / x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()

features = 512
x = torch.randn(2,10,features)
rmsnorm = RMSNorm(features)
rmsnorm(x)

tensor([[[ 0.1101, -0.0716, -0.6080,  ...,  0.4212,  0.1695,  0.2998],
         [ 0.7499,  1.0187, -0.0473,  ...,  0.3423, -0.0039, -0.6226],
         [ 1.8066, -1.2405,  1.5864,  ...,  0.5453, -0.2074,  0.2317],
         ...,
         [ 0.7046, -2.0290,  0.9100,  ..., -0.6138,  0.0429, -0.7488],
         [ 1.3223, -0.3488,  0.8212,  ..., -1.8400, -0.2028, -0.0632],
         [ 0.8199,  1.1257,  0.2228,  ..., -2.2477, -0.0320,  1.0453]],

        [[-1.2991, -0.1642, -0.1357,  ..., -1.3782, -1.6333,  0.9654],
         [ 0.8626,  0.7779,  0.6218,  ...,  0.7989,  0.3906, -0.1806],
         [-0.4618,  1.0515, -0.9387,  ...,  0.2246,  1.3415,  1.0225],
         ...,
         [ 0.5031,  1.7508, -2.1832,  ..., -1.9709, -0.1079, -0.6804],
         [-0.6517, -1.3039, -0.2936,  ...,  1.0827,  1.0004,  0.1640],
         [-0.2813,  0.5519,  0.8084,  ...,  0.1022,  0.9811,  0.6999]]],
       grad_fn=<DivBackward0>)

# 3. 交叉熵损失

In [10]:
import torch
import torch.nn.functional as F

def cross_entropy(y_pred, y_true):
    eps = 1e-9
    y_pred = torch.clamp(y_pred, eps, 1.0-eps)

    loss = -torch.sum(y_true * torch.log(y_pred))
    return loss

y_pred = torch.tensor([0.7, 0.2, 0.1])
y_true = torch.tensor([0.0, 1.0, 0.0])

cross_entropy(y_pred, y_true)

tensor(1.6094)

# Top-k

# LoRA

$\alpha/r$是缩放因子

In [ ]:
import torch.nn as nn
import torch
class LoRA(nn.Module):
    def __init__(self, in_dim, out_dim, rank = 8, alpha = 16):
        super().__init__()

        self.scale = alpha / rank
        self.A = nn.Linear(in_dim, rank, bias = False)
        self.B = nn.Linear(rank, out_dim, bias = False)

        nn.init.normal_(self.A.weight, mean=0, std=0.02)
        nn.init.zeros_(self.B.weight)
    
    def forward(self, x):
        return self.B(self.A(x)) * self.scale


# MoE

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    def __init__(self,d_model,d_hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, d_model)
        )
    def forward(self, x):
        return self.net(x)

class MoE(nn.Module):
    def __init__(self, d_model, d_hidden, num_experts=4, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        self.gate = nn.Linear(d_model, num_experts)

        self.experts = nn.ModuleList([
            Expert(d_model,d_hidden)
            for _ in range(num_experts)
        ])

    def forward(self, x):
        B, T, D = x.shape
        x_flat = x.reshape(B*T, D)
        gate_logits = self.gate(x_flat)
        gate_probs = F.softmax(gate_logits, dim=-1)

        topk_probs, topk_indices = torch.topk(
            gate_probs, self.top_k, dim=-1
        )
        topk_probs = topk_probs / topk_probs.sum(dim = -1, keepdim=True)

        output = torch.zeros_like(x_flat)

        for expert_id, expert in enumerate(self.experts):
            mask = topk_indices == expert_id # 先执行后面再执行前面
            
            if mask.any():
                token_idx, k_idx = mask.nonzero(as_tuple=True)

                expert_input = x_flat[token_idx]
                expert_output = expert(expert_input)

                weight = topk_probs[token_idx,k_idx].unsqueeze(-1)
                output[token_idx] += weight * expert_output

        return output.reshape(B,T,D)


x = torch.randn(2,10,512)
moe = MoE(512,1024,4,2)

moe(x).shape


torch.Size([2, 10, 512])